# Nanopore sequencing quality control

Use a tab-separated sequencing summary from MinKNOW, Guppy, or Dorado. The notebook summarizes read length, mean read quality and sequencing yield. It also reports the number and yield of reads that meet the thresholds selected below. Choose thresholds for your library and intended analysis.

Set the input and output paths in the next cell, then run all cells. For an R10.4.1 run, the 5mC calling script also writes a Dorado summary to calls/sequencing_summary.tsv.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SUMMARY_PATH = Path("sequencing_summary.tsv")
OUTPUT_DIR = Path("qc_output")
MIN_QSCORE = 10.0
MIN_READ_LENGTH = 1


## Load and validate the summary

Required columns are sequence_length_template and mean_qscore_template. Summaries using sequence_length or read_length for length and mean_qscore or qscore for quality are also accepted. Each row must describe one basecalled read.


In [ ]:
def summary_column(frame, options):
    for name in options:
        if name in frame:
            return name
    raise ValueError(f"Missing summary column. Expected one of: {', '.join(options)}.")


if MIN_READ_LENGTH < 1 or not np.isfinite(MIN_QSCORE) or MIN_QSCORE < 0:
    raise ValueError("Set a positive minimum length and a finite, nonnegative Q threshold.")

if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f"Set SUMMARY_PATH to your sequencing summary: {SUMMARY_PATH}")

summary = pd.read_csv(SUMMARY_PATH, sep="\t")
if summary.empty:
    raise ValueError("The sequencing summary contains no reads.")
if "read_id" in summary and (summary["read_id"].isna().any() or summary["read_id"].duplicated().any()):
    raise ValueError("Expected one row per read with a nonempty, unique read ID.")

length_column = summary_column(summary, ("sequence_length_template", "sequence_length", "read_length"))
quality_column = summary_column(summary, ("mean_qscore_template", "mean_qscore", "qscore"))
lengths = pd.to_numeric(summary[length_column], errors="raise")
qscores = pd.to_numeric(summary[quality_column], errors="raise")
if (not np.isfinite(lengths).all() or not np.isfinite(qscores).all()
        or (lengths < 0).any() or (lengths % 1 != 0).any() or (qscores < 0).any()):
    raise ValueError("Read lengths must be nonnegative integers and Q scores must be finite and nonnegative.")
lengths = lengths.astype("int64")
passing = (lengths >= MIN_READ_LENGTH) & (qscores >= MIN_QSCORE)


## Calculate QC metrics and plot the distributions

N50 is the read length at which reads of that length or longer account for at least half the sequenced bases. The report includes all reads and a separate summary of reads passing both thresholds.


In [ ]:
def read_metrics(read_lengths, read_qscores):
    total = int(read_lengths.sum())
    descending = np.sort(read_lengths.to_numpy())[::-1]
    n50 = int(descending[np.searchsorted(np.cumsum(descending), (total + 1) // 2)]) if total else 0
    return {
        "reads": len(read_lengths),
        "yield_bp": total,
        "median_length_bp": read_lengths.median(),
        "n50_bp": n50,
        "median_mean_qscore": read_qscores.median(),
    }


report = pd.DataFrame({
    "all": read_metrics(lengths, qscores),
    "pass": read_metrics(lengths[passing], qscores[passing]),
}).T.rename_axis("subset")
report["min_read_length_bp"] = MIN_READ_LENGTH
report["min_mean_qscore"] = MIN_QSCORE
report["read_fraction"] = report["reads"] / len(summary)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report.to_csv(OUTPUT_DIR / "qc_metrics.csv")
display(report)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for axis, values, threshold, label in (
    (axes[0], lengths, MIN_READ_LENGTH, "Read length (bp)"),
    (axes[1], qscores, MIN_QSCORE, "Mean read Q score"),
):
    axis.hist(values, bins=50, color="#28658a")
    axis.axvline(threshold, color="#a33b25", linestyle="--", label="Selected minimum")
    axis.set(xlabel=label, ylabel="Read count")
    axis.legend()
fig.savefig(OUTPUT_DIR / "qc_distributions.png", dpi=180)
plt.show()
